# Thai Construction Cost PDF Table Extraction (Colab)

Notebook นี้ออกแบบเพื่อดึงตารางจาก PDF ราคาวัสดุ/ค่าแรงภาษาไทย (ปี 2568) โดยเน้นความถูกต้อง, traceability และการคงข้อมูลดิบไว้สำหรับตรวจสอบย้อนหลัง

In [ ]:
!pip -q install pdfplumber pandas openpyxl xlsxwriter numpy

In [ ]:
import re
from decimal import Decimal, InvalidOperation
import pandas as pd
import pdfplumber
from google.colab import files

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError("กรุณาอัปโหลดไฟล์ PDF อย่างน้อย 1 ไฟล์")
pdf_path = next(iter(uploaded.keys()))
print(f"Using PDF: {pdf_path}")

In [ ]:
ITEM_CODE_RE = re.compile(r"^[A-Z]\d{4}$")
SECTION_RE = re.compile(r"^([A-Z])\.$")
SUBSECTION_RE = re.compile(r"^([A-Z]\d+)\.$")
HEADER_KEYWORDS = ["CODE", "รายการ", "หน่วย", "ค่าวัสดุ", "ค่าแรง", "หมายเหตุ"]
FOOTER_PATTERNS = [r"^หน้า\s*\d+", r"^Page\s*\d+", r"^หมายเหตุ", r"^ที่มา"]
TABLE_SETTINGS_LINE = {"vertical_strategy":"lines","horizontal_strategy":"lines","intersection_tolerance":5,"snap_tolerance":3,"join_tolerance":3,"edge_min_length":3,"min_words_vertical":2,"min_words_horizontal":1}
TABLE_SETTINGS_TEXT = {"vertical_strategy":"text","horizontal_strategy":"text","intersection_tolerance":6,"snap_tolerance":4,"join_tolerance":4,"min_words_vertical":1,"min_words_horizontal":1,"text_tolerance":3}
MIN_ROWS_THRESHOLD = 4

In [ ]:
def clean_text(x):
    if x is None:
        return None
    s = str(x).replace("​", " ").replace(" ", " ")
    s = re.sub(r"[ 	]+", " ", s)
    s = re.sub(r"\s*
\s*", " ", s).strip()
    return s if s else None

def is_placeholder(v):
    if v is None:
        return True
    return str(v).strip() in {"", "-", ".", "..", "..."}

def maybe_header_row(vals):
    txt = " ".join([str(v) for v in vals if v is not None]).strip().lower()
    return (sum(1 for k in HEADER_KEYWORDS if k.lower() in txt) >= 3) if txt else False

def maybe_footer_row(vals):
    txt = " ".join([str(v) for v in vals if v is not None]).strip()
    return any(re.search(p, txt, flags=re.IGNORECASE) for p in FOOTER_PATTERNS) if txt else False

def parse_numeric(v):
    raw = clean_text(v)
    if is_placeholder(raw):
        return None, raw
    s = re.sub(r"[^0-9.\-]", "", raw.replace(",", ""))
    if s in {"", "-", "."}:
        return None, raw
    try:
        d = Decimal(s)
        return (int(d) if d == d.to_integral() else float(d)), raw
    except (InvalidOperation, ValueError):
        return None, raw

def first_non_empty(*vals):
    for v in vals:
        if v is not None and str(v).strip() != "":
            return v
    return None

def normalize_row_to_six_columns(row):
    cells = [clean_text(c) for c in row] if row is not None else []
    if len(cells) == 0:
        cells = [None] * 6
    elif len(cells) < 6:
        cells = cells + [None] * (6 - len(cells))
    elif len(cells) > 6:
        code = cells[0]
        tail = cells[-4:] if len(cells) >= 5 else [None, None, None, None]
        middle = cells[1:-4] if len(cells) > 5 else cells[1:2]
        desc = " ".join([m for m in middle if m]) if middle else None
        unit, material, labor, remark = tail
        cells = [code, desc, unit, material, labor, remark]
    return {"raw_code":cells[0],"raw_description":cells[1],"raw_unit":cells[2],"raw_material_cost":cells[3],"raw_labor_cost":cells[4],"raw_remark":cells[5]}

In [ ]:
raw_records = []
page_stats = []
with pdfplumber.open(pdf_path) as pdf:
    for p_idx, page in enumerate(pdf.pages, start=1):
        extraction_mode = "line"
        used_fallback = False
        tables = page.extract_tables(table_settings=TABLE_SETTINGS_LINE) or []
        extracted_rows = [r for t in tables if t for r in t]
        if len(extracted_rows) < MIN_ROWS_THRESHOLD:
            extraction_mode = "text"
            used_fallback = True
            tables_fb = page.extract_tables(table_settings=TABLE_SETTINGS_TEXT) or []
            extracted_rows = [r for t in tables_fb if t for r in t]
        kept_on_page = skipped_header = skipped_footer = 0
        for ridx, row in enumerate(extracted_rows, start=1):
            norm = normalize_row_to_six_columns(row)
            vals = list(norm.values())
            if maybe_header_row(vals):
                skipped_header += 1
                continue
            if maybe_footer_row(vals):
                skipped_footer += 1
                continue
            if all(is_placeholder(v) for v in vals):
                continue
            raw_records.append({"source_page":p_idx,"source_row_no":ridx,**norm,"extraction_mode":extraction_mode,"used_fallback":used_fallback})
            kept_on_page += 1
        page_stats.append({"source_page":p_idx,"rows_after_extraction":len(extracted_rows),"rows_kept":kept_on_page,"skipped_header":skipped_header,"skipped_footer":skipped_footer,"extraction_mode":extraction_mode,"used_fallback":used_fallback})
raw_rows = pd.DataFrame(raw_records)
raw_rows_preclean = raw_rows.copy(deep=True)
page_summary = pd.DataFrame(page_stats)
print(f"Raw rows extracted: {len(raw_rows):,}")

In [ ]:
dataset_records = []
current_section_code = current_section_name = None
current_subsection_code = current_subsection_name = None
current_group_1 = current_group_2 = None
current_item_code = current_item_desc = None
for _, r in raw_rows.iterrows():
    source_page = int(r["source_page"]); source_row_no = int(r["source_row_no"])
    raw_code = clean_text(r.get("raw_code")); raw_desc = clean_text(r.get("raw_description"))
    raw_unit = clean_text(r.get("raw_unit")); raw_mat = clean_text(r.get("raw_material_cost"))
    raw_lab = clean_text(r.get("raw_labor_cost")); raw_remark = clean_text(r.get("raw_remark"))
    row_type = review_flag = item_code = item_description = None
    if raw_code and SECTION_RE.match(raw_code):
        current_section_code = raw_code.rstrip("."); current_section_name = raw_desc
        current_subsection_code = current_subsection_name = None
        current_group_1 = current_group_2 = None
        current_item_code = current_item_desc = None
        row_type = "review"; review_flag = "โปรดตรวจสอบ"
    elif raw_code and SUBSECTION_RE.match(raw_code):
        current_subsection_code = raw_code.rstrip("."); current_subsection_name = raw_desc
        current_group_1 = current_group_2 = None
        current_item_code = current_item_desc = None
        row_type = "review"; review_flag = "โปรดตรวจสอบ"
    elif raw_code and ITEM_CODE_RE.match(raw_code):
        row_type = "data"; item_code = raw_code; item_description = raw_desc
        current_item_code = item_code; current_item_desc = item_description
    else:
        has_value_cols = any([not is_placeholder(raw_unit), not is_placeholder(raw_mat), not is_placeholder(raw_lab), not is_placeholder(raw_remark)])
        simple_group_token = (raw_desc or "").strip().lower() in {"a","b","c","-"}
        if current_item_code and has_value_cols:
            row_type = "variant"; item_code = current_item_code; item_description = first_non_empty(raw_desc, current_item_desc)
        elif simple_group_token or (raw_desc and not has_value_cols and not raw_code):
            row_type = "review"; review_flag = "โปรดตรวจสอบ"
            if raw_desc and not current_group_1: current_group_1 = raw_desc
            elif raw_desc: current_group_2 = raw_desc
        else:
            row_type = "review"; review_flag = "โปรดตรวจสอบ"
    material_cost, material_cost_raw = parse_numeric(raw_mat)
    labor_cost, labor_cost_raw = parse_numeric(raw_lab)
    total_cost = None if (material_cost is None and labor_cost is None) else (material_cost or 0) + (labor_cost or 0)
    if row_type in {"data","variant"} and not item_description: item_description = current_item_desc
    unit = None if is_placeholder(raw_unit) else raw_unit
    remark = raw_remark
    joined_desc = " ".join([x for x in [item_description, remark] if x])
    size_match = re.search(r"(\\d+[xX×]\\d+(?:[xX×]\\d+)?(?:\\s*(?:มม\\.|mm|ซม\\.|cm|นิ้ว))?)", joined_desc or "")
    size_text = size_match.group(1) if size_match else None
    family_hint = item_code[0] if item_code else None
    search_text = " | ".join(str(x) for x in [item_code,item_description,unit,material_cost_raw,labor_cost_raw,remark,current_section_code,current_subsection_code,current_group_1,current_group_2] if x not in [None,""])
    dataset_records.append({"source_page":source_page,"source_row_no":source_row_no,"row_type":row_type,"section_code":current_section_code,"section_name":current_section_name,"subsection_code":current_subsection_code,"subsection_name":current_subsection_name,"group_1":current_group_1,"group_2":current_group_2,"item_code":item_code,"item_description":item_description,"unit":unit,"material_cost":material_cost,"labor_cost":labor_cost,"material_cost_raw":material_cost_raw,"labor_cost_raw":labor_cost_raw,"total_cost":total_cost,"remark":remark,"review_flag":review_flag,"search_text":search_text,"size_text":size_text,"family_hint":family_hint})
dataset = pd.DataFrame(dataset_records)
review_rows = dataset[dataset["review_flag"].notna()].copy()
print(dataset.head())

In [ ]:
validation_summary = pd.DataFrame([
    {"metric":"raw_rows_extracted","value":int(len(raw_rows_preclean))},
    {"metric":"dataset_rows","value":int(len(dataset))},
    {"metric":"data_rows","value":int((dataset["row_type"]=="data").sum())},
    {"metric":"variant_rows","value":int((dataset["row_type"]=="variant").sum())},
    {"metric":"review_rows","value":int((dataset["row_type"]=="review").sum())},
    {"metric":"rows_with_review_flag","value":int(dataset["review_flag"].notna().sum())},
    {"metric":"unique_item_codes","value":int(dataset["item_code"].dropna().nunique())},
    {"metric":"pages_with_raw_rows","value":int(raw_rows_preclean["source_page"].nunique() if len(raw_rows_preclean) else 0)}
])
validation_summary

In [ ]:
excel_path = "thai_cost_extraction_output.xlsx"
dataset_csv = "dataset.csv"
raw_csv = "raw_rows.csv"
with pd.ExcelWriter(excel_path, engine="xlsxwriter") as writer:
    dataset.to_excel(writer, index=False, sheet_name="dataset")
    raw_rows_preclean.to_excel(writer, index=False, sheet_name="raw_rows")
    review_rows.to_excel(writer, index=False, sheet_name="review_rows")
    page_summary.to_excel(writer, index=False, sheet_name="page_summary")
    validation_summary.to_excel(writer, index=False, sheet_name="validation_summary")
dataset.to_csv(dataset_csv, index=False, encoding="utf-8-sig")
raw_rows_preclean.to_csv(raw_csv, index=False, encoding="utf-8-sig")
print("Export complete", excel_path, dataset_csv, raw_csv)

In [ ]:
files.download("thai_cost_extraction_output.xlsx")
files.download("dataset.csv")
files.download("raw_rows.csv")